# SpamDetect AI — EDA and supervised classification

Dataset: SMS Spam Collection (UCI). This notebook must be run to produce actual empirical results.


## 1. Load and validate the dataset


In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score, average_precision_score
df=load_dataset('ucirvine/sms_spam')['train'].to_pandas().rename(columns={'sms':'message'})
print(df.shape, df.isnull().sum())
print(df.label.value_counts())
df.head()


## 2. EDA: labels, lengths and numeric patterns


In [ ]:
df=df.dropna(subset=['message','label']).copy()
conflicts=df.groupby('message').label.nunique()
assert not (conflicts>1).any(), 'Review contradictory labels'
df=df.drop_duplicates(subset=['message'])
df['characters']=df.message.str.len()
df['words']=df.message.str.split().str.len()
df['digits']=df.message.str.count(r'\d')
fig, axes=plt.subplots(1,2,figsize=(12,4))
sns.countplot(data=df,x='label',ax=axes[0]);axes[0].set_title('HAM (0) / SPAM (1)')
sns.boxplot(data=df,x='label',y='characters',ax=axes[1]);axes[1].set_title('Message length by label')
plt.tight_layout();plt.show()
df.groupby('label')[['characters','words','digits']].describe()


## 3. Stratified split, cross-validation and model selection

All TF-IDF transformations are fitted inside the training folds, preventing leakage. Test is not used to select the winning model.


In [ ]:
Xtrain,Xtest,ytrain,ytest=train_test_split(df.message,df.label.astype(int),test_size=.2,stratify=df.label,random_state=42)
models={'NB':MultinomialNB(),'LR':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42),'SVM':LinearSVC(class_weight='balanced',random_state=42)}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for name,model in models.items():
    pipe=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),max_features=10000,sublinear_tf=True)),('model',model)])
    scores=cross_validate(pipe,Xtrain,ytrain,cv=cv,scoring=['f1','precision','recall'])
    rows.append({'name':name,'f1_cv':scores['test_f1'].mean(),'precision_cv':scores['test_precision'].mean(),'recall_cv':scores['test_recall'].mean()})
summary=pd.DataFrame(rows).sort_values('f1_cv',ascending=False)
summary


## 4. Final held-out evaluation


In [ ]:
winner=summary.iloc[0]['name']
model=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),max_features=10000,sublinear_tf=True)),('model',models[winner])])
model.fit(Xtrain,ytrain)
pred=model.predict(Xtest)
score=model.predict_proba(Xtest)[:,1] if hasattr(model,'predict_proba') else model.decision_function(Xtest)
print('Selected:',winner)
print(classification_report(ytest,pred,target_names=['HAM','SPAM']))
print('ROC-AUC:',roc_auc_score(ytest,score),'Average precision:',average_precision_score(ytest,score))
ConfusionMatrixDisplay.from_predictions(ytest,pred,display_labels=['HAM','SPAM']);plt.show()


## 5. Pretrained Transformer reference (optional)

**Caution:** a pretrained spam detector may have seen the same UCI SMS dataset during its training. Do not present its score as independent test performance. Confirm the instructor-provided model link before use.


In [ ]:
# Optional demonstration (downloads weights):
# from transformers import pipeline
# reference = pipeline('text-classification', model='mrm8488/bert-tiny-finetuned-sms-spam-detection',device=-1)
# print(reference('Congratulations! You won a free prize.'))


## 6. Interpretation and conclusions

After executing, discuss the false-positive vs false-negative trade-off, dataset age, class imbalance and English-only limitation.
